# VSA-LM: Beat FlashLM SOTA on TinyStories (CUDA)

**Architecture**: AdditionLinear (L1 distance, no matmul in FFN) + LiquidCell recurrence + Capsule cognitive features

**Target**: PPL < 1.36 (FlashLM v5, 29.7M params)

**Runtime**: T4/L4 GPU via PyTorch CUDA

In [ ]:
!pip install -q datasets tokenizers
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math, os, time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name()}')

## Data Prep (ByteLevel BPE, vocab=8192)

In [ ]:
VOCAB = 8192
SEQ_LEN = 256
DATA_DIR = 'vsa_lm_data'

if not os.path.exists(f'{DATA_DIR}/tokens.npy'):
    from datasets import load_dataset
    from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

    ds = load_dataset('roneneldan/TinyStories', split='train')
    texts = [item['text'] for item in ds]
    print(f'Loaded {len(texts)} stories')

    os.makedirs(DATA_DIR, exist_ok=True)
    tok = Tokenizer(models.BPE(unk_token='<unk>'))
    tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tok.decoder = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(vocab_size=VOCAB, special_tokens=['<pad>','<unk>','<bos>','<eos>'], min_frequency=2)
    tok.train_from_iterator(texts, trainer=trainer)
    tok.save(f'{DATA_DIR}/tokenizer.json')

    all_ids = []
    for i, t in enumerate(texts):
        all_ids.extend(tok.encode(t).ids)
        if (i+1) % 200000 == 0: print(f'  {i+1}/{len(texts)}')
    tokens = np.array(all_ids, dtype=np.int32)
    np.save(f'{DATA_DIR}/tokens.npy', tokens)
    np.save(f'{DATA_DIR}/vocab.npy', np.array([tok.get_vocab_size()]))
    print(f'Done: {len(tokens)} tokens')
else:
    tokens = np.load(f'{DATA_DIR}/tokens.npy')
    print(f'Loaded {len(tokens)} tokens')

vocab = int(np.load(f'{DATA_DIR}/vocab.npy')[0])
n = len(tokens)
tr, va = tokens[:int(.8*n)], tokens[int(.8*n):int(.9*n)]

def mkseqs(t, sl):
    x, y = [], []
    for i in range(0, len(t)-sl-1, sl//2):
        x.append(t[i:i+sl]); y.append(t[i+1:i+sl+1])
    return torch.tensor(np.array(x), dtype=torch.long), torch.tensor(np.array(y), dtype=torch.long)

train_x, train_y = mkseqs(tr, SEQ_LEN)
val_x, val_y = mkseqs(va, SEQ_LEN)
print(f'Vocab={vocab}, Train={len(train_x)}, Val={len(val_x)}')

## Model: VSA-LM (PyTorch CUDA)

In [ ]:
CAPSULE_DIM = 32


class AdditionLinearCUDA(nn.Module):
    """Multiplication-free linear: output = -||W - x||_1 + bias. CUDA via torch.cdist."""
    def __init__(self, d_in, d_out):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(d_out, d_in).uniform_(-0.1, 0.1))
        self.bias = nn.Parameter(torch.zeros(d_out))
        self.d_in = d_in

    def forward(self, x):
        # x: (S, d_in) → (S, d_out)
        # -||W[i] - x[j]||_1 = -cdist(x, W, p=1)
        dist = torch.cdist(x.unsqueeze(0), self.weight.unsqueeze(0), p=1).squeeze(0)  # (S, d_out)
        return -dist + self.bias


class LiquidCellCUDA(nn.Module):
    """Continuous-time recurrence with input-dependent tau. CUDA."""
    def __init__(self, d, dt=0.02, tau_min=0.02, tau_max=2.0):
        super().__init__()
        s = math.sqrt(2.0 / (d + d))
        self.W = nn.Parameter(torch.randn(d, d) * s)
        self.U = nn.Parameter(torch.randn(d, d) * s)
        self.b = nn.Parameter(torch.zeros(d))
        self.V = nn.Parameter(torch.randn(d, d) * s)
        self.c = nn.Parameter(torch.randn(d) * 0.1)
        self.register_buffer('h', torch.zeros(d))
        self.dt = dt
        self.tau_min = tau_min
        self.tau_max = tau_max

    def step(self, x):
        # x: (d,)
        tau = self.tau_min + F.softplus(self.V @ x + self.c)
        tau = torch.clamp(tau, max=self.tau_max)
        a = torch.tanh(self.W @ self.h + self.U @ x + self.b)
        dh = -self.h / tau.clamp(min=1e-6) + a
        self.h = self.h + self.dt * dh
        return self.h

    def reset(self):
        self.h.zero_()


class VSALayerCUDA(nn.Module):
    """LiquidCell recurrence + AdditionLinear FFN + capsule modulation."""
    def __init__(self, d, d_ffn):
        super().__init__()
        self.ln = nn.LayerNorm(d)
        self.ffn_up = AdditionLinearCUDA(d, d_ffn)
        self.ffn_down = AdditionLinearCUDA(d_ffn, d)
        self.liquid = LiquidCellCUDA(d)
        self.d = d

    def forward(self, x, plasticity=0.5, consolidation=0.5):
        # x: (S, d)
        h = self.ln(x)
        x_mean = h.mean(dim=0)

        # LiquidCell
        self.liquid.dt = 0.01 + 0.03 * plasticity
        self.liquid.tau_min = 0.02 + 0.08 * consolidation
        temporal = self.liquid.step(x_mean)

        # AdditionLinear FFN (L1 distance, no matmul)
        h_up = torch.sign(self.ffn_up(h) / math.sqrt(self.ffn_up.d_in))
        h_ffn = self.ffn_down(h_up) / math.sqrt(self.ffn_down.d_in)

        # Temporal gating
        gate = torch.tanh(temporal)
        y = (0.5 + plasticity) * h_ffn * (1.0 + 0.1 * gate)
        return x + y


class VSALMModel(nn.Module):
    """VSA Language Model — CUDA."""
    def __init__(self, vocab, d, d_ffn, n_layers, max_seq):
        super().__init__()
        self.d = d
        self.embed = nn.Embedding(vocab, d)
        self.capsule_embed = nn.Embedding(vocab, CAPSULE_DIM)
        self.capsule_proj = nn.Linear(CAPSULE_DIM, d, bias=False)
        self.pos = nn.Parameter(torch.randn(max_seq + 16, d) * 0.02)
        self.out_proj = nn.Linear(d, vocab, bias=False)
        self.layers = nn.ModuleList([VSALayerCUDA(d, d_ffn) for _ in range(n_layers)])
        self.scale = math.sqrt(d)
        nn.init.normal_(self.embed.weight, 0, 0.02)
        nn.init.normal_(self.capsule_embed.weight, 0, 0.01)

    def forward(self, ids):
        # ids: (S,) long tensor
        S = ids.shape[0]
        x = self.embed(ids) + self.pos[:S]
        caps = self.capsule_embed(ids)
        x = x + self.capsule_proj(caps)

        # Capsule modulation signals
        with torch.no_grad():
            cm = caps.mean(dim=0)
            plasticity = cm[14].clamp(0, 1).item()
            consolidation = cm[18].clamp(0, 1).item()

        for layer in self.layers:
            x = layer(x, plasticity=plasticity, consolidation=consolidation)

        return self.out_proj(x / self.scale)

    def reset_liquid(self):
        for layer in self.layers:
            layer.liquid.reset()


print('Model defined.')

## Training

In [ ]:
# ── Config ──
D_MODEL = 384
N_LAYERS = 12
D_FFN = 1152
LR = 5e-4
TRAIN_STEPS = 50000
VAL_EVERY = 200
GRAD_CLIP = 1.0

# ── Init ──
model = VSALMModel(vocab, D_MODEL, D_FFN, N_LAYERS, SEQ_LEN).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'VSA-LM: d={D_MODEL}, layers={N_LAYERS}, params={n_params/1e6:.1f}M')

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, TRAIN_STEPS, eta_min=1e-5)

# ── Sanity ──
with torch.no_grad():
    test = model(train_x[0].to(device))
    print(f'Forward OK: {test.shape}, finite={torch.isfinite(test).all()}')

In [ ]:
def compute_ppl(model, x_data, y_data, max_samples=100):
    model.eval()
    total_loss, n = 0.0, 0
    with torch.no_grad():
        for i in range(min(len(x_data), max_samples)):
            logits = model(x_data[i].to(device))
            loss = F.cross_entropy(logits, y_data[i].to(device), reduction='sum')
            total_loss += loss.item()
            n += y_data[i].shape[0]
    model.train()
    return math.exp(min(total_loss / max(n, 1), 20))


t0 = time.time()
best_ppl = float('inf')
step = 0

while step < TRAIN_STEPS:
    perm = torch.randperm(len(train_x))

    for i in range(len(perm)):
        if step >= TRAIN_STEPS:
            break

        idx = perm[i]
        ids = train_x[idx].to(device)
        labels = train_y[idx].to(device)

        # Forward + loss (full backprop through AdditionLinear + LiquidCell)
        logits = model(ids)
        loss = F.cross_entropy(logits, labels)

        # Backward
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        scheduler.step()

        # Reset liquid state periodically to prevent drift
        if step % 100 == 0:
            model.reset_liquid()

        if step % VAL_EVERY == 0:
            ppl = compute_ppl(model, val_x, val_y)
            el = time.time() - t0
            sps = (step + 1) / el if el > 0 else 0
            lr_now = scheduler.get_last_lr()[0]
            print(f'step={step:6d} lr={lr_now:.5f} | loss={loss.item():.3f} | PPL={ppl:.1f} | {sps:.1f} stp/s')
            if ppl < best_ppl:
                best_ppl = ppl
                torch.save(model.state_dict(), 'vsa_lm_best.pt')

        step += 1

print(f'\nDone: {step} steps, best PPL={best_ppl:.1f}, time={time.time()-t0:.0f}s')

## Results

In [ ]:
model.load_state_dict(torch.load('vsa_lm_best.pt'))
final_ppl = compute_ppl(model, val_x, val_y, max_samples=500)

print(f"\n{'='*50}")
print(f'  VSA-LM Results')
print(f"{'='*50}")
print(f'  Final PPL:    {final_ppl:.2f}')
print(f'  Best PPL:     {best_ppl:.2f}')
print(f'  FlashLM SOTA: 1.36')
print(f'  Params:       {n_params/1e6:.1f}M')
print(f'  No MatMul:    FFN uses L1 distance (AdditionLinear)')
print(f'  Temporal:     LiquidCell continuous-time recurrence')
print(f'  Capsules:     {CAPSULE_DIM}-dim cognitive features per token')
print(f"{'='*50}")